# Scientific Image Forgery Localization — Comparable Training Pipeline

Bu notebook, bilimsel görüntülerde **pixel-level forgery localization** problemi için uçtan uca bir PyTorch segmentation pipeline'ı kurar.

## Varsayılan ilk deney
- **Model:** `DeepLabV3Plus`
- **Encoder:** `tu-efficientnet_b5`
- **Loss:** `BCEWithLogits + Dice`
- **Split:** `case_id` bazlı `GroupShuffleSplit`
- **Artifact tracking:** her çalıştırmada `runs/{run_name}/...`

## Kaydedilen çıktılar
- `config.json`
- `split_summary.json`
- `train_history.csv`
- `best_metrics.json`
- `threshold_sweep.csv`
- `best_model.pth`
- `last_model.pth`
- `oof_val_predictions.npz`
- `submission.csv`
- eğitim grafikleri ve validation görselleştirmeleri

## Notebook akışı
1. Gerekli Kütüphaneleri İçe Aktar
2. Örnek Girdi ve Beklenen Çıktıyı Tanımla
3. Temel Mantığı Fonksiyonlara Dönüştür
4. Hata Kontrolleri ve Kenar Durumları Ekle
5. Fonksiyonları Test Verileriyle Çalıştır
6. Sonucu Modüler ve Yeniden Kullanılabilir Hale Getir

In [ ]:
import os
import re
import gc
import sys
import json
import time
import random
import warnings
import subprocess
import contextlib
import importlib.util
from pathlib import Path
from datetime import datetime
from collections import defaultdict


def ensure_package(pip_name, import_name=None):
    import_name = import_name or pip_name
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


for pip_name, import_name in [
    ("segmentation-models-pytorch", "segmentation_models_pytorch"),
    ("albumentations", "albumentations"),
    ("timm", "timm"),
    ("opencv-python-headless", "cv2"),
]:
    ensure_package(pip_name, import_name)

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

from sklearn.model_selection import GroupShuffleSplit

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Örnek Girdi ve Beklenen Çıktıyı Tanımla

Beklenen veri yapısı:

```text
train_images/
  authentic/
  forged/
train_masks/
supplemental_images/
supplemental_masks/
test_images/
```

Beklenen submission formatı:
- `case_id`
- `annotation`

Boş tahminler için `annotation = "authentic"`; dolu tahminler için RLE string üretilecektir.

In [ ]:
IMG_SUFFIXES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

CFG = {
    "seed": 42,
    "data_root": "/kaggle/input/datasets/koushikkumardinda/scientific-image-forgery-detection/recodai-luc-scientific-image-forgery-detection",
    "model_name": "deeplabv3plus",          # alternatives: unetplusplus, unet
    "encoder_name": "tu-efficientnet_b5",   # easy to swap later
    "encoder_weights": "imagenet",
    "loss_name": "bce_dice",                # alternative: focal_dice
    "img_size": 512 if torch.cuda.is_available() else 320,
    "batch_size": 4 if torch.cuda.is_available() else 2,
    "epochs": 20,
    "lr": 2e-4,
    "weight_decay": 1e-4,
    "num_workers": 2 if (os.name != "nt" and torch.cuda.is_available()) else 0,
    "val_size": 0.20,
    "amp": True,
    "bce_weight": 0.4,
    "dice_weight": 0.6,
    "focal_weight": 0.3,
    "positive_sample_weight": 3.0,
    "use_weighted_sampler": True,
    "thresholds": [round(float(x), 2) for x in np.arange(0.10, 0.91, 0.05)],
    "monitor_metric": "dice",
    "early_stopping_patience": 6,
    "grad_clip": 1.0,
    "min_delta": 1e-4,
    "positive_fraction_samples": 96,
    "max_pos_weight": 50.0,
    "tta_inference": True,
    "min_component_area": 12,
    "rle_order": "C",   # change if competition expects a different flatten order
    "runs_root": None,
    "run_name": None,
}


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


seed_everything(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True


def resolve_data_root(custom_root=None):
    candidates = []
    if custom_root:
        candidates.append(Path(custom_root))

    kaggle_inputs = [Path("/kaggle/input"), Path("/kaggle/working")]
    for root in kaggle_inputs:
        if root.exists():
            candidates.extend([p for p in root.rglob("*") if p.is_dir()])

    colab_candidates = [Path("/content"), Path("/content/drive/MyDrive")]
    for root in colab_candidates:
        if root.exists():
            candidates.extend([root] + [p for p in root.rglob("*") if p.is_dir()])

    cwd = Path.cwd()
    candidates.extend([cwd, cwd.parent])

    seen = set()
    for cand in candidates:
        cand = Path(cand)
        if cand in seen:
            continue
        seen.add(cand)
        if (cand / "train_images").exists() and (cand / "test_images").exists():
            return cand
    return None


def ensure_run_directories(cfg):
    if cfg.get("runs_root"):
        runs_root = Path(cfg["runs_root"]).expanduser()
    elif Path("/kaggle/working").exists():
        runs_root = Path("/kaggle/working/runs")
    elif Path("/content").exists():
        runs_root = Path("/content/runs")
    else:
        runs_root = Path.cwd() / "runs"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_encoder = cfg["encoder_name"].replace("/", "-")
    run_name = cfg["run_name"] or f"{timestamp}_{cfg['model_name']}_{safe_encoder}_{cfg['loss_name']}"

    run_dir = runs_root / run_name
    plots_dir = run_dir / "plots"
    visualizations_dir = run_dir / "visualizations"
    for path in [runs_root, run_dir, plots_dir, visualizations_dir]:
        path.mkdir(parents=True, exist_ok=True)

    return run_name, {
        "runs_root": runs_root,
        "run_dir": run_dir,
        "plots_dir": plots_dir,
        "visualizations_dir": visualizations_dir,
        "registry_path": runs_root / "experiment_registry.csv",
        "best_model_path": run_dir / "best_model.pth",
        "last_model_path": run_dir / "last_model.pth",
        "history_path": run_dir / "train_history.csv",
        "metrics_path": run_dir / "best_metrics.json",
        "split_summary_path": run_dir / "split_summary.json",
        "threshold_path": run_dir / "threshold_sweep.csv",
        "oof_path": run_dir / "oof_val_predictions.npz",
        "submission_path": run_dir / "submission.csv",
    }


DATA_ROOT = resolve_data_root(CFG.get("data_root"))
assert DATA_ROOT is not None, "Dataset root not found. Set CFG['data_root'] manually for Kaggle/Colab/local." 

RUN_NAME, PATHS = ensure_run_directories(CFG)
CFG["data_root"] = str(DATA_ROOT)
CFG["run_name"] = RUN_NAME

with open(PATHS["run_dir"] / "config.json", "w", encoding="utf-8") as f:
    json.dump(CFG, f, indent=2)

print("DEVICE:", DEVICE)
print("DATA_ROOT:", DATA_ROOT)
print("RUN_DIR:", PATHS["run_dir"])

display(pd.DataFrame([
    {"example_case_id": 1001, "expected_annotation": "authentic"},
    {"example_case_id": 1002, "expected_annotation": "12 5 40 3 ..."},
]))

## 3. Temel Mantığı Fonksiyonlara Dönüştür

Bu bölümde veri indeksleme, maske birleştirme, metrik hesaplama ve kayıt yardımcıları tanımlanır.

## 4. Hata Kontrolleri ve Kenar Durumları Ekle

Fonksiyonlar; eksik dosya, boş veri, okunamayan görsel ve `case_id` çıkarılamayan dosya adları gibi durumlar için korumalı olacak şekilde yazılmıştır.

In [ ]:
def save_json(path, obj):
    def default(o):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        return str(o)

    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=default)


def extract_case_id_from_name(path_obj):
    stem = Path(path_obj).stem
    match = re.match(r"^(\d+)", stem)
    if not match:
        raise ValueError(f"Could not extract case_id from filename: {path_obj}")
    return int(match.group(1))


def list_image_files(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted([p for p in folder.rglob("*") if p.is_file() and p.suffix.lower() in IMG_SUFFIXES])


def read_image_rgb(path):
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")

    if img.dtype == np.uint16:
        img = cv2.convertScaleAbs(img, alpha=255.0 / 65535.0)

    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    else:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img


def load_mask_array(mask_path):
    arr = np.asarray(np.load(mask_path))
    if arr.ndim == 3:
        arr = arr[..., 0]
    arr = np.nan_to_num(arr, nan=0.0)
    return (arr > 0).astype(np.uint8)


def union_masks(mask_paths, image_shape_hw):
    h, w = image_shape_hw
    merged = np.zeros((h, w), dtype=np.uint8)
    for mask_path in mask_paths:
        mask = load_mask_array(mask_path)
        if mask.shape[:2] != (h, w):
            mask = cv2.resize(mask.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
        merged = np.maximum(merged, mask)
    return merged


def rle_encode(mask, order="C"):
    mask = (mask > 0).astype(np.uint8)
    pixels = mask.flatten(order=order)
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return "authentic" if len(runs) == 0 else " ".join(map(str, runs))


def postprocess_mask(mask, min_area=0):
    mask = (mask > 0).astype(np.uint8)
    if min_area <= 0 or mask.sum() == 0:
        return mask

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    cleaned = np.zeros_like(mask)
    for lab in range(1, num_labels):
        area = stats[lab, cv2.CC_STAT_AREA]
        if area >= min_area:
            cleaned[labels == lab] = 1
    return cleaned


def pixel_metrics_from_arrays(y_true, y_pred, eps=1e-7):
    yt = y_true.reshape(-1).astype(np.uint8)
    yp = y_pred.reshape(-1).astype(np.uint8)

    tp = np.logical_and(yt == 1, yp == 1).sum(dtype=np.float64)
    fp = np.logical_and(yt == 0, yp == 1).sum(dtype=np.float64)
    fn = np.logical_and(yt == 1, yp == 0).sum(dtype=np.float64)

    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = (2 * precision * recall) / (precision + recall + eps)
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "dice": float(dice),
        "iou": float(iou),
        "pred_pos_rate": float(yp.mean()),
        "true_pos_rate": float(yt.mean()),
    }


def soft_dice_from_probs_np(y_true, y_prob, eps=1e-7):
    y_true = y_true.reshape(-1).astype(np.float32)
    y_prob = y_prob.reshape(-1).astype(np.float32)
    inter = (y_true * y_prob).sum()
    return float((2 * inter + eps) / (y_true.sum() + y_prob.sum() + eps))


def evaluate_thresholds(all_masks, all_probs, thresholds):
    masks = all_masks[:, 0] if all_masks.ndim == 4 else all_masks
    probs = all_probs[:, 0] if all_probs.ndim == 4 else all_probs

    rows = []
    for thr in thresholds:
        preds = (probs >= thr).astype(np.uint8)
        metrics = pixel_metrics_from_arrays(masks, preds)
        metrics["threshold"] = float(thr)
        rows.append(metrics)

    sweep_df = pd.DataFrame(rows).sort_values("threshold").reset_index(drop=True)
    best_idx = sweep_df["dice"].idxmax()
    best_threshold = float(sweep_df.loc[best_idx, "threshold"])
    best_metrics = sweep_df.loc[best_idx].to_dict()
    best_metrics["soft_dice"] = float(soft_dice_from_probs_np(masks, probs))
    return sweep_df, best_threshold, best_metrics


def index_mask_files(mask_dir):
    mask_dir = Path(mask_dir)
    mask_map = defaultdict(list)
    if not mask_dir.exists():
        return mask_map
    for p in sorted(mask_dir.rglob("*.npy")):
        cid = extract_case_id_from_name(p.name)
        mask_map[cid].append(str(p))
    return mask_map


def build_samples_dataframe(data_root):
    data_root = Path(data_root)
    train_mask_map = index_mask_files(data_root / "train_masks")
    supp_mask_map = index_mask_files(data_root / "supplemental_masks")

    rows = []

    for p in list_image_files(data_root / "train_images" / "authentic"):
        cid = extract_case_id_from_name(p.name)
        rows.append({
            "case_id": cid,
            "image_path": str(p),
            "source": "train",
            "label": "authentic",
            "mask_paths": [],
            "has_mask": 0,
            "is_forged": 0,
        })

    for p in list_image_files(data_root / "train_images" / "forged"):
        cid = extract_case_id_from_name(p.name)
        mask_paths = train_mask_map.get(cid, [])
        rows.append({
            "case_id": cid,
            "image_path": str(p),
            "source": "train",
            "label": "forged",
            "mask_paths": list(mask_paths),
            "has_mask": int(len(mask_paths) > 0),
            "is_forged": 1,
        })

    for p in list_image_files(data_root / "supplemental_images"):
        cid = extract_case_id_from_name(p.name)
        mask_paths = supp_mask_map.get(cid, [])
        rows.append({
            "case_id": cid,
            "image_path": str(p),
            "source": "supplemental",
            "label": "forged" if len(mask_paths) > 0 else "authentic",
            "mask_paths": list(mask_paths),
            "has_mask": int(len(mask_paths) > 0),
            "is_forged": int(len(mask_paths) > 0),
        })

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No training samples found under: {data_root}")

    df = df.drop_duplicates(subset=["image_path"]).sort_values(["case_id", "image_path"]).reset_index(drop=True)
    missing_masks = df[(df["label"] == "forged") & (df["has_mask"] == 0)]
    if len(missing_masks) > 0:
        print(f"Warning: {len(missing_masks)} forged samples do not have a mask file. They will behave like zero-mask examples.")

    return df


def build_test_dataframe(data_root):
    data_root = Path(data_root)
    rows = []
    for p in list_image_files(data_root / "test_images"):
        rows.append({
            "case_id": extract_case_id_from_name(p.name),
            "image_path": str(p),
        })
    test_df = pd.DataFrame(rows).sort_values("case_id").reset_index(drop=True)
    if test_df.empty:
        print("No test images found; submission step will be skipped.")
    return test_df


def estimate_positive_fraction(df, max_samples=96):
    if df.empty:
        return 0.0
    sample_df = df.sample(n=min(len(df), max_samples), random_state=CFG["seed"])
    positive_pixels = 0.0
    total_pixels = 0.0
    for row in sample_df.itertuples(index=False):
        image = read_image_rgb(row.image_path)
        if row.has_mask:
            mask = union_masks(row.mask_paths, image.shape[:2])
        else:
            mask = np.zeros(image.shape[:2], dtype=np.uint8)
        positive_pixels += float(mask.sum())
        total_pixels += float(mask.size)
    return positive_pixels / max(total_pixels, 1.0)


def make_overlay(image, mask, alpha=0.35):
    image = image.copy()
    mask = (mask > 0).astype(np.uint8)
    color = np.zeros_like(image)
    color[..., 0] = 255
    return np.where(mask[..., None] > 0, (1 - alpha) * image + alpha * color, image).astype(np.uint8)


def save_history_plots(history_df, plots_dir):
    if history_df.empty:
        return

    fig, ax = plt.subplots(1, 1, figsize=(7, 4))
    ax.plot(history_df["epoch"], history_df["train_loss"], label="train_loss")
    ax.plot(history_df["epoch"], history_df["val_loss"], label="val_loss")
    ax.set_title("Loss Curve")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(Path(plots_dir) / "loss_curve.png", dpi=150)
    plt.close(fig)

    fig, ax = plt.subplots(1, 1, figsize=(7, 4))
    for col in ["val_dice", "val_iou", "val_f1"]:
        if col in history_df.columns:
            ax.plot(history_df["epoch"], history_df[col], label=col)
    ax.set_title("Validation Metrics")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(Path(plots_dir) / "metric_curve.png", dpi=150)
    plt.close(fig)


def save_threshold_plot(sweep_df, plots_dir):
    if sweep_df.empty:
        return
    fig, ax = plt.subplots(1, 1, figsize=(8, 4))
    for col in ["dice", "iou", "f1", "precision", "recall"]:
        if col in sweep_df.columns:
            ax.plot(sweep_df["threshold"], sweep_df[col], marker="o", label=col)
    ax.set_title("Threshold Sweep")
    ax.set_xlabel("Threshold")
    ax.set_ylabel("Metric")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2)
    fig.tight_layout()
    fig.savefig(Path(plots_dir) / "threshold_curve.png", dpi=150)
    plt.close(fig)

## 5. Fonksiyonları Test Verileriyle Çalıştır

Önce dataframe oluşturulacak, ardından leakage-safe split doğrulanacak ve birkaç örnek görsel kaydedilecektir.

In [ ]:
def get_preprocessing_fn(cfg):
    return smp.encoders.get_preprocessing_fn(cfg["encoder_name"], cfg["encoder_weights"])


def get_transforms(cfg, is_train=True, preprocessing_fn=None):
    transforms = [
        A.LongestMaxSize(max_size=int(cfg["img_size"])),
        A.PadIfNeeded(
            min_height=int(cfg["img_size"]),
            min_width=int(cfg["img_size"]),
            border_mode=cv2.BORDER_CONSTANT,
            value=0,
            mask_value=0,
        ),
    ]

    if is_train:
        transforms.extend([
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.05,
                scale_limit=0.10,
                rotate_limit=20,
                border_mode=cv2.BORDER_CONSTANT,
                value=0,
                mask_value=0,
                p=0.50,
            ),
            A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.35),
            A.GaussNoise(var_limit=(10.0, 40.0), p=0.15),
            A.OneOf([
                A.GaussianBlur(blur_limit=(3, 5), p=1.0),
                A.MotionBlur(blur_limit=3, p=1.0),
            ], p=0.15),
        ])

    if preprocessing_fn is not None:
        transforms.append(A.Lambda(image=lambda x, **kwargs: preprocessing_fn(x)))

    transforms.append(ToTensorV2(transpose_mask=True))
    return A.Compose(transforms)


class ForgeryDataset(Dataset):
    def __init__(self, df, cfg, is_train=True, with_masks=True):
        self.df = df.reset_index(drop=True).copy()
        self.cfg = cfg
        self.is_train = is_train
        self.with_masks = with_masks
        self.preprocessing_fn = get_preprocessing_fn(cfg)
        self.transform = get_transforms(cfg, is_train=is_train, preprocessing_fn=self.preprocessing_fn)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = read_image_rgb(row["image_path"])
        orig_h, orig_w = image.shape[:2]

        if self.with_masks and row.get("has_mask", 0):
            mask = union_masks(row["mask_paths"], image.shape[:2])
        else:
            mask = np.zeros(image.shape[:2], dtype=np.uint8)

        transformed = self.transform(image=image, mask=mask)
        image_tensor = transformed["image"].float()
        mask_tensor = transformed["mask"].float()
        if mask_tensor.ndim == 2:
            mask_tensor = mask_tensor.unsqueeze(0)
        mask_tensor = (mask_tensor > 0).float()

        return {
            "image": image_tensor,
            "mask": mask_tensor,
            "case_id": torch.tensor(int(row["case_id"]), dtype=torch.long),
            "image_path": row["image_path"],
            "orig_hw": torch.tensor([orig_h, orig_w], dtype=torch.long),
        }


def make_group_split(df, val_size=0.2, seed=42):
    splitter = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    train_idx, val_idx = next(splitter.split(df, groups=df["case_id"]))
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)
    return train_df, val_df


def validate_group_split(train_df, val_df):
    overlap = set(train_df["case_id"].tolist()).intersection(set(val_df["case_id"].tolist()))
    assert len(overlap) == 0, f"Leakage detected. Overlapping case_ids: {sorted(list(overlap))[:10]}"
    print("Leakage check passed. Overlapping case_ids:", len(overlap))


def make_dataloaders(train_df, val_df, cfg):
    train_ds = ForgeryDataset(train_df, cfg, is_train=True, with_masks=True)
    val_ds = ForgeryDataset(val_df, cfg, is_train=False, with_masks=True)

    sampler = None
    shuffle = True
    if cfg.get("use_weighted_sampler", True):
        sample_weights = np.where(train_df["is_forged"].values == 1, cfg["positive_sample_weight"], 1.0).astype(np.float64)
        sampler = WeightedRandomSampler(torch.as_tensor(sample_weights, dtype=torch.double), len(sample_weights), replacement=True)
        shuffle = False

    train_loader = DataLoader(
        train_ds,
        batch_size=int(cfg["batch_size"]),
        sampler=sampler,
        shuffle=shuffle if sampler is None else False,
        num_workers=int(cfg["num_workers"]),
        pin_memory=DEVICE.type == "cuda",
        drop_last=False,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=int(cfg["batch_size"]),
        shuffle=False,
        num_workers=int(cfg["num_workers"]),
        pin_memory=DEVICE.type == "cuda",
        drop_last=False,
    )
    return train_loader, val_loader, train_ds, val_ds


def show_dataset_samples(df, save_path, n=4):
    if df.empty:
        return
    rng = np.random.default_rng(CFG["seed"])
    pick_n = min(n, len(df))
    indices = rng.choice(len(df), size=pick_n, replace=False)

    fig, axes = plt.subplots(pick_n, 3, figsize=(12, 4 * pick_n))
    if pick_n == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_ax, idx in zip(axes, indices):
        row = df.iloc[int(idx)]
        image = read_image_rgb(row["image_path"])
        mask = union_masks(row["mask_paths"], image.shape[:2]) if row["has_mask"] else np.zeros(image.shape[:2], dtype=np.uint8)
        overlay = make_overlay(image, mask)

        row_ax[0].imshow(image)
        row_ax[0].set_title(f"case_id={row['case_id']} | {row['label']}")
        row_ax[1].imshow(mask, cmap="gray")
        row_ax[1].set_title("mask")
        row_ax[2].imshow(overlay)
        row_ax[2].set_title("overlay")
        for ax in row_ax:
            ax.axis("off")

    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.show()
    plt.close(fig)


samples_df = build_samples_dataframe(DATA_ROOT)
train_df, val_df = make_group_split(samples_df, val_size=CFG["val_size"], seed=CFG["seed"])
validate_group_split(train_df, val_df)

split_summary = {
    "n_total": int(len(samples_df)),
    "n_train": int(len(train_df)),
    "n_val": int(len(val_df)),
    "train_unique_cases": int(train_df["case_id"].nunique()),
    "val_unique_cases": int(val_df["case_id"].nunique()),
    "train_positive_images": int(train_df["is_forged"].sum()),
    "val_positive_images": int(val_df["is_forged"].sum()),
}
save_json(PATHS["split_summary_path"], split_summary)

display(pd.DataFrame([split_summary]))
display(samples_df.head())

CFG["estimated_positive_fraction"] = float(estimate_positive_fraction(train_df, max_samples=CFG["positive_fraction_samples"]))
CFG["pos_weight"] = float(min(CFG["max_pos_weight"], max(1.0, (1.0 - max(CFG["estimated_positive_fraction"], 1e-6)) / max(CFG["estimated_positive_fraction"], 1e-6))))
save_json(PATHS["run_dir"] / "config.json", CFG)

train_loader, val_loader, train_ds, val_ds = make_dataloaders(train_df, val_df, CFG)
show_dataset_samples(train_df, PATHS["visualizations_dir"] / "dataset_preview.png", n=4)

print(f"Estimated positive pixel fraction: {CFG['estimated_positive_fraction']:.8f}")
print(f"Using pos_weight: {CFG['pos_weight']:.3f}")
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 6. Sonucu Modüler ve Yeniden Kullanılabilir Hale Getir

Aşağıdaki bölüm; model factory, loss seçimi, eğitim/validasyon döngüleri, threshold tuning, artifact kaydı, test inference ve deney karşılaştırmasını tek bir yeniden kullanılabilir akışta birleştirir.

In [ ]:
class BCEDiceLoss(nn.Module):
    def __init__(self, pos_weight=1.0, bce_weight=0.4, dice_weight=0.6):
        super().__init__()
        self.register_buffer("pos_weight_tensor", torch.tensor([pos_weight], dtype=torch.float32))
        self.dice = smp.losses.DiceLoss(mode="binary", from_logits=True)
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        targets = targets.to(device=logits.device, dtype=logits.dtype)
        pos_weight = self.pos_weight_tensor.to(device=logits.device, dtype=logits.dtype)
        bce = torch.nn.functional.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight)
        dice = self.dice(logits, targets)
        return self.bce_weight * bce + self.dice_weight * dice


class FocalDiceLoss(nn.Module):
    def __init__(self, focal_weight=0.3, dice_weight=0.7):
        super().__init__()
        self.focal = smp.losses.FocalLoss(mode="binary", alpha=0.25, gamma=2.0)
        self.dice = smp.losses.DiceLoss(mode="binary", from_logits=True)
        self.focal_weight = focal_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        targets = targets.to(device=logits.device, dtype=logits.dtype)
        focal = self.focal(logits, targets)
        dice = self.dice(logits, targets)
        return self.focal_weight * focal + self.dice_weight * dice


def build_loss(cfg):
    if cfg["loss_name"].lower() == "bce_dice":
        return BCEDiceLoss(
            pos_weight=cfg["pos_weight"],
            bce_weight=cfg["bce_weight"],
            dice_weight=cfg["dice_weight"],
        )
    if cfg["loss_name"].lower() == "focal_dice":
        return FocalDiceLoss(
            focal_weight=cfg["focal_weight"],
            dice_weight=cfg["dice_weight"],
        )
    raise ValueError(f"Unsupported loss_name: {cfg['loss_name']}")


def build_model(cfg):
    common = dict(
        encoder_name=cfg["encoder_name"],
        encoder_weights=cfg["encoder_weights"],
        in_channels=3,
        classes=1,
        activation=None,
    )

    name = cfg["model_name"].lower()
    if name == "deeplabv3plus":
        model = smp.DeepLabV3Plus(**common)
    elif name == "unetplusplus":
        model = smp.UnetPlusPlus(decoder_attention_type="scse", **common)
    elif name == "unet":
        model = smp.Unet(decoder_attention_type="scse", **common)
    else:
        raise ValueError(f"Unsupported model_name: {cfg['model_name']}")

    return model


def autocast_context(device, enabled=True):
    if device.type == "cuda" and enabled:
        return torch.cuda.amp.autocast()
    return contextlib.nullcontext()


def predict_with_tta(model, images, use_tta=False):
    probs = torch.sigmoid(model(images))
    if not use_tta:
        return probs

    probs_h = torch.flip(torch.sigmoid(model(torch.flip(images, dims=[3]))), dims=[3])
    probs_v = torch.flip(torch.sigmoid(model(torch.flip(images, dims=[2]))), dims=[2])
    return (probs + probs_h + probs_v) / 3.0


def train_one_epoch(model, loader, optimizer, criterion, device, scaler, cfg, epoch):
    model.train()
    amp_enabled = bool(cfg["amp"] and device.type == "cuda")
    running_loss = 0.0
    seen = 0

    pbar = tqdm(loader, desc=f"Train {epoch:02d}", dynamic_ncols=True, leave=False)
    for batch in pbar:
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast_context(device, amp_enabled):
            logits = model(images)
            loss = criterion(logits, masks)

        if amp_enabled:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            if cfg.get("grad_clip") is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if cfg.get("grad_clip") is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            optimizer.step()

        batch_size = images.size(0)
        running_loss += float(loss.item()) * batch_size
        seen += batch_size
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / max(seen, 1)


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device, cfg, epoch):
    model.eval()
    amp_enabled = bool(cfg["amp"] and device.type == "cuda")
    running_loss = 0.0
    seen = 0
    all_probs = []
    all_masks = []
    all_paths = []

    pbar = tqdm(loader, desc=f"Valid {epoch:02d}", dynamic_ncols=True, leave=False)
    for batch in pbar:
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].to(device, non_blocking=True)

        with autocast_context(device, amp_enabled):
            logits = model(images)
            loss = criterion(logits, masks)

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        gts = masks.detach().cpu().numpy()

        batch_size = images.size(0)
        running_loss += float(loss.item()) * batch_size
        seen += batch_size

        all_probs.append(probs)
        all_masks.append(gts)
        all_paths.extend(batch["image_path"])
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    val_loss = running_loss / max(seen, 1)
    all_probs = np.concatenate(all_probs, axis=0)
    all_masks = np.concatenate(all_masks, axis=0)
    return val_loss, all_probs, all_masks, all_paths


@torch.no_grad()
def save_prediction_visualizations(model, dataset, path_bundle, device, threshold, cfg, num_samples=4):
    if len(dataset) == 0:
        return

    rng = np.random.default_rng(cfg["seed"])
    indices = rng.choice(len(dataset), size=min(num_samples, len(dataset)), replace=False)

    for i, idx in enumerate(indices, start=1):
        sample = dataset[int(idx)]
        image_tensor = sample["image"].unsqueeze(0).to(device)
        gt_mask = sample["mask"][0].cpu().numpy()
        prob = predict_with_tta(model, image_tensor, use_tta=False)[0, 0].detach().cpu().numpy()
        pred = postprocess_mask((prob >= threshold).astype(np.uint8), min_area=cfg.get("min_component_area", 0))

        raw_image = read_image_rgb(sample["image_path"])
        raw_image = cv2.resize(raw_image, (prob.shape[1], prob.shape[0]), interpolation=cv2.INTER_AREA)
        pred_overlay = make_overlay(raw_image, pred)

        fig, axes = plt.subplots(1, 5, figsize=(18, 4))
        axes[0].imshow(raw_image)
        axes[0].set_title("image")
        axes[1].imshow(gt_mask, cmap="gray")
        axes[1].set_title("ground truth")
        axes[2].imshow(prob, cmap="magma")
        axes[2].set_title("probability")
        axes[3].imshow(pred, cmap="gray")
        axes[3].set_title(f"prediction @ {threshold:.2f}")
        axes[4].imshow(pred_overlay)
        axes[4].set_title("overlay")
        for ax in axes:
            ax.axis("off")
        fig.tight_layout()
        fig.savefig(path_bundle["visualizations_dir"] / f"val_sample_{i:03d}.png", dpi=150)
        plt.show()
        plt.close(fig)


@torch.no_grad()
def make_submission(model, data_root, cfg, path_bundle, device, threshold):
    test_df = build_test_dataframe(data_root)
    if test_df.empty:
        return pd.DataFrame(columns=["case_id", "annotation"])

    test_df = test_df.copy()
    test_df["mask_paths"] = [[] for _ in range(len(test_df))]
    test_df["has_mask"] = 0
    test_df["is_forged"] = 0
    test_df["label"] = "test"

    test_ds = ForgeryDataset(test_df, cfg, is_train=False, with_masks=False)
    test_loader = DataLoader(
        test_ds,
        batch_size=int(cfg["batch_size"]),
        shuffle=False,
        num_workers=int(cfg["num_workers"]),
        pin_memory=device.type == "cuda",
    )

    model.eval()
    rows = []
    for batch in tqdm(test_loader, desc="Inference", dynamic_ncols=True):
        images = batch["image"].to(device, non_blocking=True)
        probs = predict_with_tta(model, images, use_tta=cfg.get("tta_inference", True)).detach().cpu().numpy()

        for i in range(images.size(0)):
            case_id = int(batch["case_id"][i].item())
            orig_h, orig_w = batch["orig_hw"][i].tolist()
            prob = probs[i, 0]
            prob = cv2.resize(prob, (int(orig_w), int(orig_h)), interpolation=cv2.INTER_LINEAR)
            pred = postprocess_mask((prob >= threshold).astype(np.uint8), min_area=cfg.get("min_component_area", 0))
            rows.append({
                "case_id": case_id,
                "annotation": rle_encode(pred, order=cfg.get("rle_order", "C")),
            })

    submission = pd.DataFrame(rows).sort_values("case_id").reset_index(drop=True)
    submission.to_csv(path_bundle["submission_path"], index=False)
    return submission


def update_experiment_registry(registry_path, row_dict):
    registry_path = Path(registry_path)
    if registry_path.exists():
        registry_df = pd.read_csv(registry_path)
        registry_df = registry_df[registry_df["run_name"] != row_dict["run_name"]].copy()
    else:
        registry_df = pd.DataFrame()

    registry_df = pd.concat([registry_df, pd.DataFrame([row_dict])], ignore_index=True)
    if "val_dice" in registry_df.columns:
        registry_df = registry_df.sort_values("val_dice", ascending=False).reset_index(drop=True)
    registry_df.to_csv(registry_path, index=False)
    return registry_df


def run_experiment(cfg, train_loader, val_loader, train_ds, val_ds, data_root, path_bundle, device):
    model = build_model(cfg).to(device)
    criterion = build_loss(cfg).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    scaler = torch.cuda.amp.GradScaler(enabled=bool(cfg["amp"] and device.type == "cuda"))

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Model: {cfg['model_name']} | Encoder: {cfg['encoder_name']} | Trainable params: {n_params:,}")

    history_rows = []
    best_score = -np.inf
    best_payload = None
    best_sweep_df = pd.DataFrame()
    best_val_probs = None
    best_val_masks = None
    patience_counter = 0
    start_time = time.time()

    for epoch in range(1, int(cfg["epochs"]) + 1):
        epoch_start = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler, cfg, epoch)
        val_loss, val_probs, val_masks, _ = validate_one_epoch(model, val_loader, criterion, device, cfg, epoch)

        sweep_df, tuned_threshold, metrics = evaluate_thresholds(val_masks, val_probs, cfg["thresholds"])
        current_lr = float(optimizer.param_groups[0]["lr"])

        history_row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "lr": current_lr,
            "best_threshold": tuned_threshold,
            "val_dice": metrics["dice"],
            "val_iou": metrics["iou"],
            "val_f1": metrics["f1"],
            "val_precision": metrics["precision"],
            "val_recall": metrics["recall"],
            "soft_dice": metrics["soft_dice"],
            "epoch_minutes": (time.time() - epoch_start) / 60.0,
        }
        history_rows.append(history_row)
        history_df = pd.DataFrame(history_rows)
        history_df.to_csv(path_bundle["history_path"], index=False)

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "cfg": cfg,
            "metrics": metrics,
            "threshold": tuned_threshold,
        }
        torch.save(checkpoint, path_bundle["last_model_path"])

        monitor_value = float(metrics[cfg["monitor_metric"]])
        scheduler.step(monitor_value)

        print(
            f"Epoch {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
            f"dice={metrics['dice']:.4f} | iou={metrics['iou']:.4f} | thr={tuned_threshold:.2f} | lr={current_lr:.2e}"
        )

        if monitor_value > best_score + cfg.get("min_delta", 0.0):
            best_score = monitor_value
            patience_counter = 0
            best_sweep_df = sweep_df.copy()
            best_val_probs = val_probs.copy()
            best_val_masks = val_masks.copy()
            best_payload = {
                "run_name": cfg["run_name"],
                "best_epoch": epoch,
                "model_name": cfg["model_name"],
                "encoder_name": cfg["encoder_name"],
                "loss_name": cfg["loss_name"],
                "best_threshold": float(tuned_threshold),
                "val_dice": float(metrics["dice"]),
                "val_iou": float(metrics["iou"]),
                "val_f1": float(metrics["f1"]),
                "val_precision": float(metrics["precision"]),
                "val_recall": float(metrics["recall"]),
                "soft_dice": float(metrics["soft_dice"]),
                "checkpoint_path": str(path_bundle["best_model_path"]),
            }
            save_json(path_bundle["metrics_path"], best_payload)
            best_sweep_df.to_csv(path_bundle["threshold_path"], index=False)
            np.savez_compressed(path_bundle["oof_path"], probs=best_val_probs, masks=best_val_masks)
            torch.save(checkpoint, path_bundle["best_model_path"])
        else:
            patience_counter += 1

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        gc.collect()

        if patience_counter >= int(cfg["early_stopping_patience"]):
            print(f"Early stopping triggered at epoch {epoch}.")
            break

    history_df = pd.DataFrame(history_rows)
    save_history_plots(history_df, path_bundle["plots_dir"])
    if not best_sweep_df.empty:
        save_threshold_plot(best_sweep_df, path_bundle["plots_dir"])

    if path_bundle["best_model_path"].exists():
        best_ckpt = torch.load(path_bundle["best_model_path"], map_location=device)
        model.load_state_dict(best_ckpt["model_state_dict"])

    best_threshold = float(best_payload["best_threshold"]) if best_payload else 0.5
    save_prediction_visualizations(model, val_ds, path_bundle, device, best_threshold, cfg, num_samples=4)
    submission_df = make_submission(model, data_root, cfg, path_bundle, device, best_threshold)

    total_minutes = (time.time() - start_time) / 60.0
    if best_payload is not None:
        best_payload["train_time_minutes"] = total_minutes
        save_json(path_bundle["metrics_path"], best_payload)

    registry_row = {
        "run_name": cfg["run_name"],
        "model_name": cfg["model_name"],
        "encoder_name": cfg["encoder_name"],
        "image_size": cfg["img_size"],
        "batch_size": cfg["batch_size"],
        "loss_name": cfg["loss_name"],
        "best_epoch": int(best_payload["best_epoch"] if best_payload else -1),
        "best_threshold": float(best_threshold),
        "val_dice": float(best_payload["val_dice"] if best_payload else np.nan),
        "val_iou": float(best_payload["val_iou"] if best_payload else np.nan),
        "val_f1": float(best_payload["val_f1"] if best_payload else np.nan),
        "val_precision": float(best_payload["val_precision"] if best_payload else np.nan),
        "val_recall": float(best_payload["val_recall"] if best_payload else np.nan),
        "train_time_minutes": float(total_minutes),
        "checkpoint_path": str(path_bundle["best_model_path"]),
    }
    registry_df = update_experiment_registry(path_bundle["registry_path"], registry_row)

    return {
        "history_df": history_df,
        "best_metrics": best_payload,
        "submission_df": submission_df,
        "registry_df": registry_df,
        "run_dir": str(path_bundle["run_dir"]),
    }


results = run_experiment(CFG, train_loader, val_loader, train_ds, val_ds, DATA_ROOT, PATHS, DEVICE)
print("Artifacts saved under:", results["run_dir"])
if results["best_metrics"] is not None:
    display(pd.DataFrame([results["best_metrics"]]))
if not results["submission_df"].empty:
    display(results["submission_df"].head())

if PATHS["registry_path"].exists():
    compare_df = pd.read_csv(PATHS["registry_path"])
    display(compare_df.sort_values("val_dice", ascending=False).reset_index(drop=True))

In [ ]:
# 0.75 eşiği elle sabitlenmedi; validation threshold sweep içinde en yüksek Dice'ı verdiği için seçildi.
# Aşağıdaki hücre, en iyi checkpoint'i diskten yükleyip validation split içindeki gerçek maskesi olan forged örnekler için çıktıları gösterir.

@torch.no_grad()
def visualize_forged_validation_examples_from_best_checkpoint(
    val_df,
    cfg,
    device,
    checkpoint_path=None,
    num_samples=6,
    threshold=None,
    compare_threshold=0.50,
    use_tta=False,
):
    forged_val_df = val_df[val_df["has_mask"] == 1].reset_index(drop=True)
    if forged_val_df.empty:
        forged_val_df = val_df[val_df["is_forged"] == 1].reset_index(drop=True)
        print("Not: Validation split içinde mask dosyası bulunan forged örnek yok; is_forged==1 örnekleri gösteriliyor.")
    if forged_val_df.empty:
        print("Validation split içinde forged örnek bulunamadı.")
        return None

    ckpt_candidates = []
    if checkpoint_path is not None:
        ckpt_candidates.append(Path(checkpoint_path))
    if "PATHS" in globals():
        ckpt_candidates.append(Path(PATHS["best_model_path"]))
    ckpt_candidates.extend([
        Path.cwd() / "deeplabv3+" / "best_model.pth",
        Path.cwd() / "best_model.pth",
    ])

    ckpt_path = next((p for p in ckpt_candidates if p.exists()), None)
    if ckpt_path is None:
        raise FileNotFoundError(
            "best_model.pth bulunamadı. checkpoint_path verin veya dosyayı çalışma klasörüne koyun."
        )

    ckpt = torch.load(ckpt_path, map_location=device)
    model_cfg = dict(cfg)
    if isinstance(ckpt, dict) and "cfg" in ckpt and isinstance(ckpt["cfg"], dict):
        model_cfg.update(ckpt["cfg"])

    model = build_model(model_cfg).to(device)
    state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    model.load_state_dict(state_dict)
    model.eval()

    if threshold is None:
        threshold = float(
            ckpt.get(
                "threshold",
                globals().get("results", {}).get("best_metrics", {}).get("best_threshold", 0.5),
            )
        )

    if "PATHS" in globals():
        save_dir = Path(PATHS["visualizations_dir"]) / "best_ckpt_forged_examples"
    else:
        save_dir = Path.cwd() / "best_ckpt_forged_examples"
    save_dir.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(model_cfg.get("seed", 42))
    pick_n = min(int(num_samples), len(forged_val_df))
    chosen_indices = rng.choice(len(forged_val_df), size=pick_n, replace=False)

    shown_case_ids = []
    print(f"Checkpoint: {ckpt_path}")
    print(f"Best threshold: {threshold:.2f} | Compare threshold: {compare_threshold:.2f}")
    print(f"Forged validation sample count: {len(forged_val_df)} | Showing: {pick_n}")
    print(f"Saved figures -> {save_dir}")

    for vis_idx, row_idx in enumerate(chosen_indices, start=1):
        row = forged_val_df.iloc[int(row_idx)]
        shown_case_ids.append(int(row["case_id"]))

        image = read_image_rgb(row["image_path"])
        gt_mask = union_masks(row["mask_paths"], image.shape[:2]) if row["has_mask"] else np.zeros(image.shape[:2], dtype=np.uint8)

        one_item_ds = ForgeryDataset(pd.DataFrame([row]), model_cfg, is_train=False, with_masks=True)
        sample = one_item_ds[0]
        image_tensor = sample["image"].unsqueeze(0).to(device)

        prob = predict_with_tta(model, image_tensor, use_tta=use_tta)[0, 0].detach().cpu().numpy()
        prob = cv2.resize(prob, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_LINEAR)

        pred_best = postprocess_mask((prob >= threshold).astype(np.uint8), min_area=model_cfg.get("min_component_area", 0))
        pred_mid = postprocess_mask((prob >= compare_threshold).astype(np.uint8), min_area=model_cfg.get("min_component_area", 0))

        fig, axes = plt.subplots(1, 6, figsize=(22, 4))
        axes[0].imshow(image)
        axes[0].set_title(f"image | case_id={row['case_id']}")
        axes[1].imshow(gt_mask, cmap="gray")
        axes[1].set_title("ground truth")
        axes[2].imshow(prob, cmap="magma")
        axes[2].set_title("probability")
        axes[3].imshow(pred_best, cmap="gray")
        axes[3].set_title(f"pred @ {threshold:.2f}")
        axes[4].imshow(pred_mid, cmap="gray")
        axes[4].set_title(f"pred @ {compare_threshold:.2f}")
        axes[5].imshow(make_overlay(image, pred_best))
        axes[5].set_title("overlay @ best_thr")

        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        fig.savefig(save_dir / f"best_ckpt_forged_{vis_idx:03d}_case_{int(row['case_id'])}.png", dpi=150)
        plt.show()
        plt.close(fig)

    return {
        "save_dir": str(save_dir),
        "shown_case_ids": shown_case_ids,
        "best_threshold": float(threshold),
    }


best_vis_info = visualize_forged_validation_examples_from_best_checkpoint(
    val_df=val_df,
    cfg=CFG,
    device=DEVICE,
    checkpoint_path=Path.cwd() / "deeplabv3+" / "best_model.pth",
    num_samples=6,
    threshold=None,
    compare_threshold=0.50,
    use_tta=CFG.get("tta_inference", False),
)

best_vis_info

In [ ]:
# Hücre 1 — forged örneklerde ground truth / maske kontrolü

def inspect_forged_ground_truth_status(df, split_name="dataset", max_rows=20):
    forged_df = df[df["label"] == "forged"].copy().reset_index(drop=True)
    if forged_df.empty:
        print(f"{split_name} içinde forged örnek bulunamadı.")
        return pd.DataFrame(), pd.DataFrame()

    gt_positive_pixels = []
    gt_is_empty = []
    n_mask_files = []

    for row in forged_df.itertuples(index=False):
        n_mask_files.append(len(row.mask_paths))
        if row.has_mask:
            image = read_image_rgb(row.image_path)
            gt_mask = union_masks(row.mask_paths, image.shape[:2])
            pos_pixels = int(gt_mask.sum())
        else:
            pos_pixels = 0
        gt_positive_pixels.append(pos_pixels)
        gt_is_empty.append(int(pos_pixels == 0))

    forged_df["n_mask_files"] = n_mask_files
    forged_df["gt_positive_pixels"] = gt_positive_pixels
    forged_df["gt_is_empty"] = gt_is_empty

    summary_df = pd.DataFrame([
        {
            "split": split_name,
            "forged_total": int(len(forged_df)),
            "forged_with_mask_file": int((forged_df["has_mask"] == 1).sum()),
            "forged_without_mask_file": int((forged_df["has_mask"] == 0).sum()),
            "forged_with_nonempty_gt": int((forged_df["gt_positive_pixels"] > 0).sum()),
            "forged_with_empty_gt": int((forged_df["gt_positive_pixels"] == 0).sum()),
        }
    ])

    print(f"\n[{split_name}] forged ground truth kontrol özeti")
    display(summary_df)

    suspicious_df = forged_df[
        (forged_df["has_mask"] == 0) | (forged_df["gt_positive_pixels"] == 0)
    ][[
        "case_id", "source", "label", "has_mask", "n_mask_files", "gt_positive_pixels", "image_path"
    ]].sort_values(["has_mask", "gt_positive_pixels", "case_id"]).reset_index(drop=True)

    if suspicious_df.empty:
        print("Şüpheli kayıt yok: forged örneklerin ground truth maskeleri dolu görünüyor.")
    else:
        print("Ground truth'u siyah çıkabilecek forged örnekler:")
        display(suspicious_df.head(max_rows))

    return summary_df, forged_df


train_forged_summary_df, train_forged_audit_df = inspect_forged_ground_truth_status(
    samples_df,
    split_name="all_samples",
    max_rows=20,
)

val_forged_summary_df, val_forged_audit_df = inspect_forged_ground_truth_status(
    val_df,
    split_name="validation",
    max_rows=20,
)

# Hücre 2 — maskesi bulunan forged validation görüntülerde en iyi ağırlıklarla tahmin

@torch.no_grad()
def predict_on_masked_forged_validation_with_best_ckpt(
    val_df,
    cfg,
    device,
    checkpoint_path=None,
    num_samples=6,
    threshold=None,
    compare_threshold=0.50,
    only_nonempty_gt=True,
    use_tta=False,
):
    candidate_df = val_df[(val_df["label"] == "forged") & (val_df["has_mask"] == 1)].copy().reset_index(drop=True)
    if candidate_df.empty:
        print("Validation split içinde maskesi bulunan forged örnek yok.")
        return None

    if only_nonempty_gt:
        positive_keep = []
        for _, row in candidate_df.iterrows():
            image = read_image_rgb(row["image_path"])
            gt_mask = union_masks(row["mask_paths"], image.shape[:2])
            positive_keep.append(int(gt_mask.sum()) > 0)
        candidate_df = candidate_df[np.array(positive_keep, dtype=bool)].reset_index(drop=True)

    if candidate_df.empty:
        print("Mask dosyası olan ama ground truth'u boş olmayan forged validation örneği bulunamadı.")
        return None

    ckpt_candidates = []
    if checkpoint_path is not None:
        ckpt_candidates.append(Path(checkpoint_path))
    if "PATHS" in globals():
        ckpt_candidates.append(Path(PATHS["best_model_path"]))
    ckpt_candidates.extend([
        Path.cwd() / "deeplabv3+" / "best_model.pth",
        Path.cwd() / "best_model.pth",
    ])

    ckpt_path = next((p for p in ckpt_candidates if p.exists()), None)
    if ckpt_path is None:
        raise FileNotFoundError("best_model.pth bulunamadı.")

    ckpt = torch.load(ckpt_path, map_location=device)
    model_cfg = dict(cfg)
    if isinstance(ckpt, dict) and "cfg" in ckpt and isinstance(ckpt["cfg"], dict):
        model_cfg.update(ckpt["cfg"])

    model = build_model(model_cfg).to(device)
    state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
    model.load_state_dict(state_dict)
    model.eval()

    if threshold is None:
        threshold = float(ckpt.get("threshold", 0.5)) if isinstance(ckpt, dict) else 0.5

    save_dir = Path(PATHS["visualizations_dir"]) / "best_ckpt_masked_forged_only"
    save_dir.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(model_cfg.get("seed", 42))
    pick_n = min(int(num_samples), len(candidate_df))
    chosen_indices = rng.choice(len(candidate_df), size=pick_n, replace=False)
    shown_case_ids = []

    print(f"Checkpoint: {ckpt_path}")
    print(f"Using threshold: {threshold:.2f} | Compare threshold: {compare_threshold:.2f}")
    print(f"Eligible masked forged validation samples: {len(candidate_df)} | Showing: {pick_n}")
    print(f"Saved figures -> {save_dir}")

    for vis_idx, row_idx in enumerate(chosen_indices, start=1):
        row = candidate_df.iloc[int(row_idx)]
        shown_case_ids.append(int(row["case_id"]))

        image = read_image_rgb(row["image_path"])
        gt_mask = union_masks(row["mask_paths"], image.shape[:2])

        one_item_ds = ForgeryDataset(pd.DataFrame([row]), model_cfg, is_train=False, with_masks=True)
        sample = one_item_ds[0]
        image_tensor = sample["image"].unsqueeze(0).to(device)

        prob = predict_with_tta(model, image_tensor, use_tta=use_tta)[0, 0].detach().cpu().numpy()
        prob = cv2.resize(prob, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_LINEAR)

        pred_best = postprocess_mask((prob >= threshold).astype(np.uint8), min_area=model_cfg.get("min_component_area", 0))
        pred_cmp = postprocess_mask((prob >= compare_threshold).astype(np.uint8), min_area=model_cfg.get("min_component_area", 0))

        fig, axes = plt.subplots(1, 6, figsize=(22, 4))
        axes[0].imshow(image)
        axes[0].set_title(f"image | case_id={row['case_id']}")
        axes[1].imshow(gt_mask, cmap="gray")
        axes[1].set_title("ground truth")
        axes[2].imshow(prob, cmap="magma")
        axes[2].set_title("probability")
        axes[3].imshow(pred_best, cmap="gray")
        axes[3].set_title(f"pred @ {threshold:.2f}")
        axes[4].imshow(pred_cmp, cmap="gray")
        axes[4].set_title(f"pred @ {compare_threshold:.2f}")
        axes[5].imshow(make_overlay(image, pred_best))
        axes[5].set_title("overlay @ best_thr")

        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        fig.savefig(save_dir / f"masked_forged_val_{vis_idx:03d}_case_{int(row['case_id'])}.png", dpi=150)
        plt.show()
        plt.close(fig)

    return {
        "save_dir": str(save_dir),
        "shown_case_ids": shown_case_ids,
        "threshold": float(threshold),
    }


masked_forged_pred_info = predict_on_masked_forged_validation_with_best_ckpt(
    val_df=val_df,
    cfg=CFG,
    device=DEVICE,
    checkpoint_path=Path.cwd() / "deeplabv3+" / "best_model.pth",
    num_samples=6,
    threshold=None,
    compare_threshold=0.50,
    only_nonempty_gt=True,
    use_tta=CFG.get("tta_inference", False),
)

masked_forged_pred_info

In [ ]:
# Segmentation için filtrelenmiş train/val split:
# authentic negatifleri KORU, forged örneklerden ise sadece non-empty GT maskesi olanları kullan.


def annotate_ground_truth_columns(df):
    df = df.copy().reset_index(drop=True)
    gt_positive_pixels = []
    gt_is_empty = []
    has_nonempty_gt = []
    n_mask_files = []

    for row in df.itertuples(index=False):
        mask_paths = list(row.mask_paths) if isinstance(row.mask_paths, (list, tuple)) else []
        n_mask_files.append(len(mask_paths))

        pos_pixels = 0
        if getattr(row, "has_mask", 0) and len(mask_paths) > 0:
            image = read_image_rgb(row.image_path)
            gt_mask = union_masks(mask_paths, image.shape[:2])
            pos_pixels = int(gt_mask.sum())

        gt_positive_pixels.append(pos_pixels)
        gt_is_empty.append(int(pos_pixels == 0))
        has_nonempty_gt.append(int(pos_pixels > 0))

    df["n_mask_files"] = n_mask_files
    df["gt_positive_pixels"] = gt_positive_pixels
    df["gt_is_empty"] = gt_is_empty
    df["has_nonempty_gt"] = has_nonempty_gt
    return df


if "samples_df" not in globals():
    samples_df = build_samples_dataframe(DATA_ROOT)

samples_df = annotate_ground_truth_columns(samples_df)
min_gt_positive_pixels = int(CFG.get("min_gt_positive_pixels", 1))

# Keep: authentic negatives + forged samples with non-empty masks
seg_samples_df = samples_df[
    (samples_df["label"] == "authentic") |
    (samples_df["gt_positive_pixels"] >= min_gt_positive_pixels)
].copy().reset_index(drop=True)

dropped_empty_forged_df = samples_df[
    (samples_df["label"] == "forged") &
    (samples_df["gt_positive_pixels"] < min_gt_positive_pixels)
].copy().reset_index(drop=True)

if seg_samples_df.empty:
    raise RuntimeError("Filtered segmentation dataset is empty. No non-empty GT examples were found.")

train_df_filtered, val_df_filtered = make_group_split(
    seg_samples_df,
    val_size=CFG["val_size"],
    seed=CFG["seed"],
)
validate_group_split(train_df_filtered, val_df_filtered)

filtered_split_summary = {
    "n_total_all_samples": int(len(samples_df)),
    "n_used_for_segmentation": int(len(seg_samples_df)),
    "n_dropped_empty_gt_forged": int(len(dropped_empty_forged_df)),
    "train_samples": int(len(train_df_filtered)),
    "val_samples": int(len(val_df_filtered)),
    "train_authentic": int((train_df_filtered["label"] == "authentic").sum()),
    "train_nonempty_gt_forged": int((train_df_filtered["gt_positive_pixels"] > 0).sum()),
    "val_authentic": int((val_df_filtered["label"] == "authentic").sum()),
    "val_nonempty_gt_forged": int((val_df_filtered["gt_positive_pixels"] > 0).sum()),
}

display(pd.DataFrame([filtered_split_summary]))
display(seg_samples_df.head())

filtered_cfg = dict(CFG)
filtered_cfg["train_only_on_nonempty_gt"] = True
filtered_cfg["min_gt_positive_pixels"] = min_gt_positive_pixels
filtered_cfg["estimated_positive_fraction"] = float(
    estimate_positive_fraction(train_df_filtered, max_samples=filtered_cfg["positive_fraction_samples"])
)
filtered_cfg["pos_weight"] = float(
    min(
        filtered_cfg["max_pos_weight"],
        max(
            1.0,
            (1.0 - max(filtered_cfg["estimated_positive_fraction"], 1e-6)) /
            max(filtered_cfg["estimated_positive_fraction"], 1e-6),
        ),
    )
)

train_loader_filtered, val_loader_filtered, train_ds_filtered, val_ds_filtered = make_dataloaders(
    train_df_filtered,
    val_df_filtered,
    filtered_cfg,
)

preview_df = train_df_filtered[train_df_filtered["gt_positive_pixels"] > 0].copy().reset_index(drop=True)
if not preview_df.empty:
    show_dataset_samples(preview_df, PATHS["visualizations_dir"] / "filtered_nonempty_gt_preview.png", n=min(4, len(preview_df)))

print("Filtered segmentation training is ready.")
print(f"Dropped empty-GT forged samples: {len(dropped_empty_forged_df)}")
print(f"Estimated positive pixel fraction: {filtered_cfg['estimated_positive_fraction']:.8f}")
print(f"Using pos_weight: {filtered_cfg['pos_weight']:.3f}")
print(f"Filtered train batches: {len(train_loader_filtered)} | Filtered val batches: {len(val_loader_filtered)}")

In [ ]:
# Filtrelenmiş split ile yeniden eğitim (authentic + non-empty GT forged)

filtered_cfg = dict(filtered_cfg)
filtered_cfg["run_name"] = (
    f"{datetime.now().strftime('%Y%m%d_%H%M%S')}_"
    f"{filtered_cfg['model_name']}_{filtered_cfg['encoder_name'].replace('/', '-')}_"
    f"{filtered_cfg['loss_name']}_nonemptyGT"
)

FILTERED_RUN_NAME, FILTERED_PATHS = ensure_run_directories(filtered_cfg)
filtered_cfg["run_name"] = FILTERED_RUN_NAME
filtered_cfg["data_root"] = str(DATA_ROOT)

save_json(FILTERED_PATHS["run_dir"] / "config.json", filtered_cfg)

filtered_results = run_experiment(
    filtered_cfg,
    train_loader_filtered,
    val_loader_filtered,
    train_ds_filtered,
    val_ds_filtered,
    DATA_ROOT,
    FILTERED_PATHS,
    DEVICE,
)

print("Filtered artifacts saved under:", filtered_results["run_dir"])
if filtered_results["best_metrics"] is not None:
    display(pd.DataFrame([filtered_results["best_metrics"]]))
if not filtered_results["submission_df"].empty:
    display(filtered_results["submission_df"].head())

filtered_results